In [ ]:

%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import numpy as np
import mne
from conditioning.utils import get_info, mkfilt_eloreta_v2
from conditioning.utils import get_mne_src_fwd_inv


Assume that the pre analyzed Data is in the data/ folder, ready to be analyzed by MNE

We provide the results for Subjects fsaverage and OAS1_0004_MR1 in the repository, so running this script is not necessary

In [ ]:
fs_dict = get_mne_src_fwd_inv("train")
def generete_necessary_data(subject, mode="train"):
    subjects_dir = "data/"
    info = get_info()
    vol_src = mne.setup_volume_source_space(subject, pos=7,
                                                subjects_dir=subjects_dir, verbose=False)
    try:
        bem = mne.read_bem_solution(f"{subjects_dir}/{subject}/BEM{mode}.fif")
    except:
        if mode == "train":
            conductivity = [0.3, 0.006, 0.3]
        else:
            conductivity = [0.332, 0.0113, 0.332]
        bem_init = mne.make_bem_model(subject,subjects_dir=subjects_dir, conductivity=conductivity, verbose=False)
        bem = mne.make_bem_solution(bem_init, verbose=False)
        mne.write_bem_solution(f"{subjects_dir}/{subject}/BEM{mode}.fif", bem, verbose=False)

    trans = subjects_dir+"/"+subject+"/HeadTransform.fif"
    try:
        fwd = mne.read_forward_solution(f"{subjects_dir}/{subject}/forward{mode}-fwd.fif")
    except:
        fwd = mne.make_forward_solution(info, trans=trans, src=vol_src,
                                            bem=bem, eeg=True, meg=False, mindist=5.0, n_jobs=1, verbose=False)
        mne.write_forward_solution(f"{subjects_dir}/{subject}/forward{mode}-fwd.fif", fwd)
    sens_pos = np.array(list(info.get_montage()._get_ch_pos().values()))

    pinv = mkfilt_eloreta_v2(fwd["sol"]["data"].reshape(61, -1, 3))

    if subject != "fsaverage":
        # We also load the fs average src to learn a morphing:
        #fs_dict = get_mne_src_fwd_inv("train")

        # Morph fsaverage source space to new subject
        morph = mne.compute_source_morph(
            src=fs_dict["fwd"]["src"],
            subject_from="fsaverage",
            subject_to=subject,
            subjects_dir=subjects_dir,
            src_to=fwd["src"],
        )
        morph.compute_vol_morph_mat()
        morph_matrix = morph.vol_morph_mat
    else:
        morph_matrix = None
    train_data = {"LeadField": fwd["sol"]["data"][:,:,None],
                  "PseudoInv": pinv,
                  "grid_loc": fwd["source_rr"],
                  "sens_loc": sens_pos,
                  "source_inds": fwd["src"][0]["vertno"],
                  "source_mask": fwd["src"][0]["inuse"].reshape(29,29,29),
                  "morph": morph_matrix
                  }
    np.save(f"{subjects_dir}/{subject}/HeadInformation{mode}-fwd.npy", train_data)
    return train_data

subjects = ["OAS1_0001_MR1", "OAS1_0002_MR1", "OAS004"]
for subject in subjects:
    generete_necessary_data(subject,mode="train")
    generete_necessary_data(subject,mode="val")
